# Foundation MLIPs as ASE calculators (MACE-MP & GRACE-FS)

Demonstrates that the `Engine` Protocol composes with **any** ASE calculator, including the recent universal/foundation interatomic potentials:

- [**MACE-MP**](https://github.com/ACEsuit/mace) — `mace.calculators.mace_mp(...)`
- [**GRACE-FS**](https://gracemaker.readthedocs.io) — `tensorpotential.calculator.grace_fm(...)`

Both ship ASE-compatible calculator wrappers, so the *only* line that changes between EMT, MACE, and GRACE is the `calculator=` argument to `ASEEngine`. The physics task description (vacancy formation energy on a small FCC Al supercell, below) is identical across all three backends.

If MACE / GRACE aren't installed, the corresponding cells skip with an `Install with: ...` hint so the notebook still runs end-to-end.

In [1]:
import os, warnings, logging

# Quiet down TensorFlow / e3nn / pyiron-workflow startup chatter.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

from ase.build import bulk
from ase.calculators.emt import EMT

from pyiron_workflow_atomistics.engine import ASEEngine, CalcInputMinimize
from pyiron_workflow_atomistics.physics.point_defect import get_vacancy_formation_energy

al_bulk = bulk("Al", "fcc", a=4.05, cubic=True)
print(f"Al bulk: {al_bulk.get_chemical_formula()}, n={len(al_bulk)}")

Al bulk: Al4, n=4


## Helper: build an ASEEngine + vacancy-formation workflow from any calculator

The same `get_vacancy_formation_energy` macro is reused with each backend; only the `calculator=` field of the engine changes.

In [2]:
def vacancy_e_form(calculator, label, workdir):
    engine = ASEEngine(
        EngineInput=CalcInputMinimize(force_convergence_tolerance=0.05, max_iterations=100),
        calculator=calculator,
        working_directory=workdir,
    )
    wf = get_vacancy_formation_energy(
        structure=al_bulk,
        engine=engine,
        min_dimensions=[8, 8, 8],  # ~3x3x3 FCC Al primitive ~ 108 atoms
    )
    wf.run()
    e_f = wf.outputs.vacancy_formation_energy.value
    print(f"{label:8s}  E_vac_form(Al FCC) = {e_f:.3f} eV")
    return e_f

## 1. EMT (classical, fast reference)

EMT is a coarse but-always-available potential for Al/Cu/Ni/Pd/Ag/Pt/Au. It under-predicts vacancy energies versus DFT.

In [3]:
e_emt = vacancy_e_form(EMT(), label="EMT", workdir="./_fm_emt")

      Step     Time          Energy          fmax
BFGS:    0 17:51:22        0.926900        0.067070


BFGS:    1 17:51:22        0.926106        0.063977


BFGS:    2 17:51:22        0.918670        0.019346


      Step     Time          Energy          fmax
BFGS:    0 17:51:22       -0.048066        0.000000


EMT       E_vac_form(Al FCC) = 0.965 eV


## 2. MACE-MP (universal foundation model)

`mace_mp(model="small")` downloads the universal MACE-MP-0 small foundation potential on first use (~30 MB) and caches it under `~/.cache/mace/`. Trained on Materials Project — works for any element combination in MP.

Install: `pip install mace-torch` (or `uv pip install mace-torch`).

In [4]:
try:
    from mace.calculators import mace_mp
    mace_calc = mace_mp(model="small", default_dtype="float64", device="cpu")
    e_mace = vacancy_e_form(mace_calc, label="MACE-MP", workdir="./_fm_mace")
except ImportError:
    e_mace = None
    print("MACE not installed - skipping.\nInstall with: pip install mace-torch  (or `uv pip install mace-torch`)")

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


Using Materials Project MACE for MACECalculator with /home/liger/.cache/mace/20231210mace128L0_energy_epoch249model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


      Step     Time          Energy          fmax
BFGS:    0 17:51:29     -114.379872        0.134386


BFGS:    1 17:51:30     -114.382950        0.126796


BFGS:    2 17:51:31     -114.407101        0.029141


      Step     Time          Energy          fmax
BFGS:    0 17:51:31     -118.706220        0.000000


MACE-MP   E_vac_form(Al FCC) = 0.590 eV


## 3. GRACE-FS (universal foundation model)

`grace_fm("GRACE-FS-OAM")` downloads the GRACE-FS foundation potential on first use and caches it under `~/.cache/grace/`. Trained on OMat24 + sAlex + MPTraj — typically faster than MACE-MP on CPU thanks to a compact graph formulation.

Install: `pip install tensorpotential` (or `uv pip install tensorpotential`).

In [5]:
try:
    from tensorpotential.calculator import grace_fm
    grace_calc = grace_fm("GRACE-FS-OAM")
    e_grace = vacancy_e_form(grace_calc, label="GRACE-FS", workdir="./_fm_grace")
except ImportError:
    e_grace = None
    print("GRACE / tensorpotential not installed - skipping.\nInstall with: pip install tensorpotential  (or `uv pip install tensorpotential`)")

[tensorpotential] Info: Environment variable TF_USE_LEGACY_KERAS is automatically set to '1'.


E0000 00:00:1778601091.957808  144665 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778601091.963574  144665 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778601091.981449  144665 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778601091.981486  144665 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778601091.981489  144665 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778601091.981491  144665 computation_placer.cc:177] computation placer already registered. Please check linka

Using cached GRACE model from /home/liger/.cache/grace/GRACE-FS-OAM
Model license: Academic Software License


I0000 00:00:1778601099.077412  144665 service.cc:152] XLA service 0x585292a7c610 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778601099.077558  144665 service.cc:160]   StreamExecutor device (0): Host, Default Version


      Step     Time          Energy          fmax
BFGS:    0 17:51:43     -115.172399        0.128413


I0000 00:00:1778601103.909415  144665 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


BFGS:    1 17:51:44     -115.175619        0.122301


BFGS:    2 17:51:44     -115.204123        0.023990


      Step     Time          Energy          fmax
BFGS:    0 17:51:49     -119.204487        0.000000


GRACE-FS  E_vac_form(Al FCC) = 0.275 eV


## Summary

Same `get_vacancy_formation_energy` macro, same `al_bulk` structure, three calculators. Foundation MLIPs (MACE, GRACE) target DFT-PBE quality (~0.6 eV for Al vacancy in PBE); EMT systematically under-predicts.

To plug in your own potential — Quip GAP, NequIP, Allegro, an ASE-wrapped DFT code, anything — just hand its ASE calculator to `ASEEngine(calculator=...)` and reuse the rest.

In [6]:
print(f"{'backend':10s} {'E_vac (eV)':>12s}")
print("-" * 23)
for label, val in [("EMT", e_emt), ("MACE-MP", e_mace), ("GRACE-FS", e_grace)]:
    print(f"{label:10s} {val if val is not None else 'skipped':>12}")

backend      E_vac (eV)
-----------------------
EMT        0.9652331842740964
MACE-MP    0.5895501101056055
GRACE-FS   0.27522344315511305
